# Práctica guiada: formatos de datos en Big Data con PySpark

En esta práctica vas a trabajar con los formatos de datos más habituales en ingeniería de datos (**CSV, JSON/JSONL, Avro, Arrow/Feather, Parquet y ORC**) usando **PySpark en Databricks Free Edition**. 

Partimos de un caso de negocio: el departamento de analítica de una empresa de distribución recibe cada día un fichero de **ventas** en CSV. El fichero no deja de crecer y el equipo tiene que decidir en qué formato guardarlo, cómo comprimirlo y cómo organizarlo para que las consultas sean rápidas y baratas. Para responder con datos y no con opiniones, vas a escribir el mismo dataset en cada formato y comparar los resultados.

**Antes de empezar:**

1. Importa este notebook en tu workspace (*Workspace → Import → File*) y conéctalo al cómputo **Serverless**, el único disponible en Free Edition.
2. Ejecuta las celdas **en orden**. Las secciones dependen de las variables que crean las anteriores.
3. No necesitas descargar nada: en la Sección 1 generarás tú mismo un dataset sintético de ventas que reproduce el fichero original de los apuntes (`pdi_sales.csv`).

## Sección 0. Preparación del entorno

Databricks Free Edition no permite escribir en la raíz de DBFS (`dbfs:/`). Para guardar ficheros usaremos un **volumen de Unity Catalog**, que se comporta como una carpeta accesible desde tres sitios:

| Desde… | Ruta | Ejemplo |
|---|---|---|
| Spark | `/Volumes/<catálogo>/<esquema>/<volumen>/...` | `spark.read.parquet("/Volumes/workspace/default/formatos_datos/ventas")` |
| Python estándar (`open`, `os`, PyArrow, Pandas) | la misma ruta | `open("/Volumes/workspace/default/formatos_datos/pdi_sales.csv")` |
| `dbutils.fs` | la misma ruta | `dbutils.fs.ls("/Volumes/workspace/default/formatos_datos")` |

Que la misma ruta funcione en Spark y en Python nos permite **escribir con Spark e inspeccionar los bytes con PyArrow o con `open()`**, algo que haremos constantemente.

Primero instalamos `fastavro`, que usaremos únicamente para inspeccionar por dentro los ficheros Avro que genere Spark, y `cramjam`, que `fastavro` necesita para descomprimir los bloques comprimidos con *Snappy* (el códec que Spark usa por defecto en Avro). Tras instalar librerías hay que reiniciar el intérprete de Python.

In [ ]:
%pip install fastavro cramjam --quiet

In [ ]:
dbutils.library.restartPython()

Ahora creamos el volumen. En Free Edition ya existen el catálogo `workspace` y el esquema `default`. Si tu instructor te indica otros nombres, cámbialos aquí.

In [ ]:
CATALOGO = "workspace"
ESQUEMA = "default"
VOLUMEN = "formatos_datos"

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.{ESQUEMA}.{VOLUMEN}")

BASE = f"/Volumes/{CATALOGO}/{ESQUEMA}/{VOLUMEN}"
print(f"Trabajaremos en: {BASE}")

Definimos unas funciones auxiliares que usaremos durante toda la práctica:

- `tamano_bytes(ruta)`: suma el tamaño de los ficheros de datos de una ruta. Spark no escribe un fichero, sino **un directorio** con uno o varios ficheros `part-...` y ficheros de control (`_SUCCESS`, `_committed_...`, `_started_...`). Estos últimos se ignoran.
- `ficheros_datos(ruta, extension)`: devuelve los ficheros de datos de un directorio.
- `arbol(ruta)`: dibuja el árbol de carpetas.
- `plan_scan(df)`: muestra solo las líneas del plan de ejecución que indican qué lee Spark del disco (`ReadSchema`) y qué filtros entrega al lector (líneas que contienen `Filters`). En *Serverless* las consultas las ejecuta el motor **Photon**, cuyos planes pueden usar nombres algo distintos (`PhotonScan`, `RequiredDataFilters`, `DictionaryFilters`...). Si alguna vez la salida te parece incompleta, usa `plan_scan(df, completo=True)` para ver el plan entero.

In [ ]:
import os, glob, json, time, math, io, contextlib
from pyspark.sql import functions as F
from pyspark.sql import types as T

def tamano_bytes(ruta):
    """Tamaño en bytes de los ficheros de datos (ignora _SUCCESS, _committed_, .crc...)."""
    if os.path.isfile(ruta):
        return os.path.getsize(ruta)
    total = 0
    for raiz, _, ficheros in os.walk(ruta):
        for f in ficheros:
            if not f.startswith(("_", ".")):
                total += os.path.getsize(os.path.join(raiz, f))
    return total

def mib(n_bytes):
    return n_bytes / 1024**2

def ficheros_datos(ruta, extension=""):
    """Ficheros de datos de un directorio escrito por Spark."""
    return sorted(p for p in glob.glob(f"{ruta}/**/*{extension}", recursive=True)
                  if os.path.isfile(p) and not os.path.basename(p).startswith(("_", ".")))

def arbol(ruta, max_ficheros=2):
    """Dibuja el árbol de directorios mostrando, como mucho, max_ficheros por carpeta."""
    for raiz, dirs, ficheros in os.walk(ruta):
        dirs.sort()
        nivel = raiz[len(ruta):].count(os.sep)
        print("    " * nivel + os.path.basename(raiz) + "/")
        visibles = sorted(f for f in ficheros if not f.startswith(("_", ".")))
        for f in visibles[:max_ficheros]:
            print("    " * (nivel + 1) + f)
        if len(visibles) > max_ficheros:
            print("    " * (nivel + 1) + f"... ({len(visibles) - max_ficheros} más)")

def plan_scan(df, completo=False):
    """Muestra las líneas del plan físico relacionadas con la lectura de datos
    (ReadSchema y cualquier línea de filtros: PushedFilters, PartitionFilters, DataFilters...).
    Con completo=True imprime el plan entero."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain("formatted")
    plan = buffer.getvalue()
    lineas = [l.strip()[:160] for l in plan.splitlines()
              if ("ReadSchema" in l or "Filters" in l) and "[]" not in l]
    print("\n".join(dict.fromkeys(lineas)) if lineas and not completo else plan)

print("Funciones auxiliares listas")

## Sección 1. Generación del dataset sintético de ventas

Los apuntes anteriores trabajan con el fichero `pdi_sales.csv`, con unas **841.000 ventas** de cuatro países. Como no tenemos el fichero original, vamos a generar uno con la misma estructura y las mismas "imperfecciones", porque precisamente esas imperfecciones son las que nos obligarán a limpiar y tipar los datos:

```text
ProductID;Date;Zip;Units;Revenue;Country
725;1/15/1999;41540   ;1;115.5;Germany
787;6/15/2002;41540   ;1;314.9;Germany
609;2/15/1999;M4E     ;1;778.8;Canada
```

| Columna | Tipo | Particularidad |
|---|---|---|
| `ProductID` | entero | Entre 1 y 2412 |
| `Date` | texto | Formato estadounidense `M/d/yyyy`, siempre día 15, entre 1999 y 2015 |
| `Zip` | texto | Código postal **con espacios a la derecha**. Canadá usa códigos alfanuméricos |
| `Units` | entero | Casi siempre 1 (columna muy repetitiva) |
| `Revenue` | decimal | Importe de la venta |
| `Country` | texto | Solo 4 valores: Canada, France, Germany, Mexico (baja cardinalidad) |
| — | — | El separador es **punto y coma** (`;`) |

Generamos los datos con funciones de Spark (`F.rand` con semilla), así que el proceso se ejecuta en paralelo y es reproducible: todos obtendréis datos muy parecidos.

In [ ]:
N_FILAS = 841_147
SEMILLA = 42
LETRAS = "ABCEGHJKLMNPRSTVXY"   # letras válidas en los códigos postales canadienses

pais = (F.when(F.col("r_pais") < 0.35, "Germany")
         .when(F.col("r_pais") < 0.65, "France")
         .when(F.col("r_pais") < 0.85, "Mexico")
         .otherwise("Canada"))

# Unos 5.000 códigos postales distintos por país (como en el fichero real, se repiten mucho)
zip_numerico = (F.floor(F.col("r_zip") * 5_000) * 17 + 1_000).cast("int").cast("string")
zip_canada = F.concat(
    F.lit(LETRAS).substr((F.col("r_zip") * 18).cast("int") + 1, F.lit(1)),
    ((F.col("r_zip") * 1_000) % 10).cast("int").cast("string"),
    F.lit(LETRAS).substr(((F.col("r_zip") * 10_000) % 18).cast("int") + 1, F.lit(1)),
)
codigo_postal = (F.when(F.col("Country").isin("Germany", "France"), F.lpad(zip_numerico, 5, "0"))
                  .when(F.col("Country") == "Mexico", zip_numerico)
                  .otherwise(zip_canada))

ventas_sinteticas = (
    spark.range(N_FILAS)
    .withColumn("r_pais", F.rand(SEMILLA))
    .withColumn("r_zip", F.rand(SEMILLA + 1))
    .withColumn("Country", pais)
    .withColumn("ProductID", (F.rand(SEMILLA + 2) * 2412).cast("int") + 1)
    .withColumn("Date", F.date_format(
        F.add_months(F.to_date(F.lit("1999-01-15")), (F.rand(SEMILLA + 3) * 198).cast("int")),
        "M/d/yyyy"))
    .withColumn("Zip", F.rpad(codigo_postal, 8, " "))          # espacios a la derecha
    .withColumn("Units", F.when(F.rand(SEMILLA + 4) < 0.9, F.lit(1))
                          .otherwise((F.rand(SEMILLA + 5) * 9).cast("int") + 2))
    .withColumn("Revenue", F.round((F.rand(SEMILLA + 6) * 990 + 10) * F.col("Units"), 1))
    .select("ProductID", "Date", "Zip", "Units", "Revenue", "Country")
)

display(ventas_sinteticas.limit(10))

Ahora lo guardamos como **un único fichero CSV** llamado `pdi_sales.csv`, igual que el original.

Fíjate en dos detalles:

- Spark siempre escribe un **directorio**. Para obtener un único fichero, escribimos con `coalesce(1)` en una carpeta temporal y copiamos el `part-...csv` resultante con `dbutils.fs.cp`.
- Al **escribir** CSV, Spark elimina por defecto los espacios al principio y al final de cada valor. Para conservar los espacios del código postal hay que desactivar esa opción con `ignoreTrailingWhiteSpace = False`.

In [ ]:
RUTA_CSV = f"{BASE}/pdi_sales.csv"
RUTA_TMP = f"{BASE}/_tmp_csv"

(ventas_sinteticas
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .option("sep", ";")
    .option("ignoreTrailingWhiteSpace", False)   # conservamos los espacios del Zip
    .csv(RUTA_TMP))

if os.path.exists(RUTA_CSV):                             # por si repetimos la práctica
    dbutils.fs.rm(RUTA_CSV)
dbutils.fs.cp(ficheros_datos(RUTA_TMP, ".csv")[0], RUTA_CSV)
dbutils.fs.rm(RUTA_TMP, True)

print(f"{RUTA_CSV}: {mib(tamano_bytes(RUTA_CSV)):.2f} MiB\n")
with open(RUTA_CSV) as f:
    for _ in range(6):
        print(repr(f.readline()))

Usamos `repr()` para que se vean los espacios del código postal y el salto de línea `\n` del final de cada registro.

A partir de aquí **olvidamos cómo se generó el fichero** y lo tratamos como un fichero recibido de un sistema externo. Lo leemos **declarando el esquema**, en vez de pedirle a Spark que lo adivine con `inferSchema` (más adelante veremos por qué es importante):

In [ ]:
esquema_ventas = T.StructType([
    T.StructField("ProductID", T.IntegerType()),
    T.StructField("Date",      T.StringType()),
    T.StructField("Zip",       T.StringType()),
    T.StructField("Units",     T.IntegerType()),
    T.StructField("Revenue",   T.DoubleType()),
    T.StructField("Country",   T.StringType()),
])

ventas_raw = (spark.read
              .option("header", True)
              .option("sep", ";")
              .schema(esquema_ventas)
              .csv(RUTA_CSV))

# Limpieza: quitamos los espacios del Zip y convertimos la fecha en un tipo DATE de verdad
ventas = (ventas_raw
          .withColumn("Zip", F.trim("Zip"))
          .withColumn("Date", F.to_date("Date", "M/d/yyyy")))

ventas.printSchema()
print(f"Filas: {ventas.count():,}")
display(ventas.groupBy("Country").count().orderBy("Country"))

## Sección 2. ¿Qué le pedimos a un formato de datos?

Conforme los datos recorren los *pipelines*, el ingeniero de datos tiene que serializarlos en distintos formatos y convertir unos en otros. Un buen formato debería ser:

- **Independiente del lenguaje**: lo escribe Java y lo lee Python sin problemas.
- **Expresivo**: admite estructuras complejas y anidadas.
- **Eficiente**: rápido de leer y escribir, y compacto en disco.
- **Dinámico**: los programas pueden procesar tipos de datos nuevos.
- **Autodescriptivo (*standalone*)**: el fichero contiene todo lo necesario para interpretarlo.
- **Divisible (*splittable*) y comprimible**: Spark, Trino, DuckDB o Hadoop solo pueden repartir un fichero entre varios nodos si se puede **trocear en fragmentos independientes**.

Según cómo almacenan los datos, los formatos se dividen en:

- **Texto** (CSV, XML, JSON): legibles e interoperables, pero lentos y voluminosos.
- **Binario** (Avro, Parquet, ORC): más rápidos y compactos, pero no se pueden abrir con un editor.

| Característica | CSV | JSON | JSONL | Avro | Parquet | ORC |
|---|:-:|:-:|:-:|:-:|:-:|:-:|
| Legible por humanos | ✅ | ✅ | ✅ | | | |
| Independiente del lenguaje | ✅ | ✅ | ✅ | ✅ | ✅ | ✅ |
| Expresivo (datos anidados) | | ✅ | ✅ | ✅ | ✅ | ✅ |
| Eficiente en tamaño | | | | ✅ | ✅ | ✅ |
| Autodescriptivo | ❔ | ✅ | ✅ | ✅ | ✅ | ✅ |
| Divisible | ❔ | | ✅ | ✅ | ✅ | ✅ |
| Orientado a columnas | | | | | ✅ | ✅ |
| *Pushdown* de predicados | | | | | ✅ | ✅ |
| Evolución de esquema | | ❔ | ❔ | ✅ | ❔ | ❔ |
| Uso típico | Intercambio | APIs | Logs, eventos | *Streaming* | Analítica | Analítica (Hive) |

Dos matices importantes:

- **CSV divisible ❔**: un CSV en texto plano se puede dividir, pero si se comprime con *gzip* deja de serlo. Avro, Parquet y ORC siguen siendo divisibles aunque estén comprimidos, porque comprimen **por bloques** (bloque, *row group* y *stripe*, respectivamente).
- **Avro autodescriptivo ✅**: es cierto para el fichero `.avro`, que lleva el esquema en la cabecera. Los mensajes Avro que viajan por Kafka con *Schema Registry* solo llevan un identificador del esquema, así que por sí solos no son autodescriptivos.

Elegir bien el formato aporta mejor rendimiento de lectura y escritura, ficheros divisibles, soporte para esquemas que evolucionan y compresión.

### 2.1. JSON frente a JSONL

- Un documento **JSON** es un único objeto que puede ocupar muchas líneas. Para leer cualquier registro hay que *parsear* el documento entero, así que **no es divisible**.
- **JSONL** (*JSON Lines*) guarda **un objeto JSON por línea**. Cada línea es independiente, así que el fichero se puede trocear por saltos de línea.

```json
{ "empleados": [
    {"nombre": "Aitor", "altura": 182, "edad": 48},
    {"nombre": "Pedro", "altura": 178, "edad": null}
  ]
}
```

```json
{"nombre": "Aitor", "altura": 182, "edad": 48}
{"nombre": "Pedro", "altura": 178, "edad": null}
```

Esta distinción tiene una consecuencia práctica en Spark que sorprende a mucha gente: **`df.write.json()` escribe JSONL, no JSON**, y **`spark.read.json()` espera JSONL**. Vamos a comprobarlo creando un pequeño DataFrame de empleados que reutilizaremos en toda la práctica:

In [ ]:
esquema_empleados = T.StructType([
    T.StructField("nombre", T.StringType(),  nullable=False),
    T.StructField("altura", T.IntegerType(), nullable=False),
    T.StructField("edad",   T.IntegerType(), nullable=True),   # Pedro no tiene edad
])

empleados = spark.createDataFrame(
    [("Aitor", 182, 48), ("Pedro", 178, None)],
    schema=esquema_empleados,
)

RUTA_EMP_JSON = f"{BASE}/empleados_json"
empleados.coalesce(1).write.mode("overwrite").json(RUTA_EMP_JSON)

fichero = ficheros_datos(RUTA_EMP_JSON, ".json")[0]
print(f"Fichero generado: {os.path.basename(fichero)}\n")
print(open(fichero).read())

Spark ha escrito un objeto por línea (JSONL). Además, **ha omitido la clave `edad` de Pedro** porque su valor es nulo: por defecto Spark no escribe los campos nulos en JSON.

Ahora hagamos lo contrario: escribimos a mano el documento JSON "clásico" de varias líneas e intentamos leerlo con Spark.

In [ ]:
RUTA_EMP_JSON_CLASICO = f"{BASE}/empleados_clasico.json"

documento = """{ "empleados": [
    {"nombre": "Aitor", "altura": 182, "edad": 48},
    {"nombre": "Pedro", "altura": 178, "edad": null}
  ]
}"""
with open(RUTA_EMP_JSON_CLASICO, "w") as f:
    f.write(documento)

# 1. Lectura por defecto: Spark espera un objeto por línea
print("Lectura por defecto (JSONL):")
por_lineas = spark.read.json(RUTA_EMP_JSON_CLASICO)
por_lineas.printSchema()
display(por_lineas)

# 2. Lectura indicando que el documento ocupa varias líneas
print("Lectura con multiLine=True:")
clasico = spark.read.option("multiLine", True).json(RUTA_EMP_JSON_CLASICO)
clasico.printSchema()
display(clasico.select(F.explode("empleados").alias("e")).select("e.*"))

En la lectura por defecto, Spark trata **cada línea como un registro independiente**. Las líneas de los empleados las interpreta como si fueran registros sueltos, y las líneas que no son un objeto completo (`{ "empleados": [`, `]`, `}`) acaban en la columna interna `_corrupt_record`. El resultado mezcla datos y basura, y no avisa con ningún error. Con `multiLine=True` lo lee, pero **todo el documento lo procesa una única tarea**, porque no se puede dividir. Con un JSON de varios GB eso es un cuello de botella.

> **El precio de no tener esquema.** Guardar JSON como texto es cómodo, pero cada consulta tiene que *parsear* el documento completo. La alternativa clásica, aplanar el JSON en columnas tipadas, es rápida, pero cualquier cambio en el origen rompe el *pipeline*. Hay una tercera vía moderna: el tipo **VARIANT**, incorporado a la especificación de Parquet en 2025 y disponible en Databricks. Lo veremos en la siguiente sesión, sobre SQL analítico.

#### Reto 2.1

Lee el directorio `RUTA_EMP_JSON` (el JSONL que escribió Spark) **sin indicar esquema** y compara los tipos que infiere Spark con los de `esquema_empleados`. ¿Qué tipo asigna a `altura` y `edad`? ¿Qué implicaciones tiene en el tamaño de los datos si después se guardan en un formato binario?

In [ ]:
# Reto 2.1 · Leemos el JSONL SIN esquema y comparamos con el esquema declarado
import pyarrow.parquet as pq   # (se importa también más adelante, en la Sección 3.2)

emp_inferido = spark.read.json(RUTA_EMP_JSON)

print("Esquema INFERIDO por Spark:")
emp_inferido.printSchema()
print("Esquema DECLARADO (esquema_empleados):")
empleados.printSchema()

# ¿Qué pasa al guardar ambos en un formato binario? Lo comprobamos con Parquet
RUTA_R21_INF = f"{BASE}/reto21_inferido"
RUTA_R21_DEC = f"{BASE}/reto21_declarado"
emp_inferido.coalesce(1).write.mode("overwrite").parquet(RUTA_R21_INF)
empleados.coalesce(1).write.mode("overwrite").parquet(RUTA_R21_DEC)

print("\nParquet con esquema inferido:")
print(pq.read_schema(ficheros_datos(RUTA_R21_INF, ".parquet")[0]))
print("\nParquet con esquema declarado:")
print(pq.read_schema(ficheros_datos(RUTA_R21_DEC, ".parquet")[0]))

# Conclusión:
# - Spark infiere `bigint` (64 bits) para altura y edad, porque en JSON un entero no tiene tamaño y elige el más seguro.
#   Además reordena las columnas alfabéticamente (altura, edad, nombre) y marca todo como nullable.
# - El esquema declarado usa `int` (32 bits). Con un esquema inferido cada entero ocupa el doble en formatos
#   binarios (int64 frente a int32) y, en millones de filas, eso se nota en disco y en memoria.
#   (Parquet recodifica y comprime, así que la diferencia real en disco puede ser menor, pero el tipo queda
#   mal definido en el contrato). Por eso en producción se declara siempre el esquema.

## Sección 3. Filas frente a columnas

CSV, XML o JSON son **formatos orientados a filas**: cada registro se guarda completo, uno detrás de otro. En un **formato orientado a columnas**, todos los valores de una columna se guardan juntos.

Tomemos tres pedidos de una tabla `orders`:

| order_id | cliente | importe | pais |
|---|---|---|---|
| 1 | Acme | 120.5 | ES |
| 2 | Globex | 80.0 | ES |
| 3 | Initech | 310.2 | FR |

Así se dispondrían en disco con cada organización:

```text
Por filas:     1,Acme,120.5,ES | 2,Globex,80.0,ES | 3,Initech,310.2,FR
Por columnas:  1,2,3 | Acme,Globex,Initech | 120.5,80.0,310.2 | ES,ES,FR
```

Si la consulta es `SELECT SUM(importe) FROM orders`:

- **Por filas**, hay que leer todos los bytes del fichero y descartar tres de cada cuatro valores.
- **Por columnas**, basta con leer el bloque `120.5,80.0,310.2`.

Además, como los valores de una columna son del mismo tipo y suelen repetirse (`ES,ES,FR`), **se comprimen mucho mejor**.

El almacenamiento por columnas también tiene inconvenientes:

- Para reconstruir un registro completo hay que leer de varios bloques de columnas.
- Actualizar una fila supone descomprimir, modificar, volver a comprimir y reescribir cada columna afectada. Para limitarlo, las columnas se reparten en varios ficheros usando particionado y *clustering*, pero modificar una sola fila sigue siendo caro.

Por eso las bases de datos **transaccionales (OLTP)** almacenan por filas y los sistemas **analíticos (OLAP)** almacenan por columnas.

### 3.1. Codificando las columnas

Antes de comprimir, los formatos columnares **recodifican** cada columna aprovechando que todos sus valores son del mismo tipo y que suelen repetirse. Las técnicas más habituales son:

- **Codificación por diccionario** (*dictionary encoding*): si una columna tiene pocos valores distintos (**baja cardinalidad**), se guarda un diccionario con los valores únicos y cada dato se sustituye por su índice.
  `Germany, Germany, France, Germany` → diccionario `{0: Germany, 1: France}` + datos `0, 0, 1, 0`
- **Run-Length Encoding (RLE)**: los valores consecutivos repetidos se guardan como pares (valor, repeticiones). `0, 0, 0, 1, 1, 0` → `(0×3), (1×2), (0×1)`. **Cuanto más ordenados estén los datos, mejor funciona.**
- ***Bit-packing***: los índices del diccionario son números pequeños, así que no necesitan 32 o 64 bits. Con 4 países bastan **2 bits** por valor.
- **Codificación delta**: en columnas crecientes (fechas, *timestamps*, identificadores) se guarda el primer valor y después solo las diferencias, que son números pequeños.

> **Codificación ≠ compresión.** Son dos capas en cascada. La **codificación** conoce el tipo de dato y actúa columna a columna. Algunos motores pueden incluso filtrar sobre los índices del diccionario sin descodificarlos. La **compresión** (*Snappy*, *gzip*, *zstd*...) es genérica: trata los bytes ya codificados como una secuencia opaca. Por eso un CSV con *gzip* sigue ocupando más que un Parquet con *Snappy*: el CSV no puede codificar por columnas.

¿Qué columnas de nuestras ventas se beneficiarán más de la codificación por diccionario? Calculemos su **cardinalidad**, es decir, el número de valores distintos de cada una:

In [ ]:
cardinalidad = ventas.agg(*[F.countDistinct(c).alias(c) for c in ventas.columns])
display(cardinalidad)

`Country` tiene 4 valores distintos entre más de 800.000 filas, y `Units` muy pocos. Son las candidatas perfectas para diccionario + RLE. `Revenue` y `Zip`, en cambio, tienen muchos valores distintos.

Simulemos en Python lo que haría un formato columnar con la columna `Country`, tomando 30 valores **tal y como vienen** y esos mismos 30 valores **ordenados**:

In [ ]:
def diccionario(valores):
    dicc = {v: i for i, v in enumerate(dict.fromkeys(valores))}
    return dicc, [dicc[v] for v in valores]

def rle(indices):
    pares, actual, cuenta = [], indices[0], 0
    for v in indices:
        if v == actual:
            cuenta += 1
        else:
            pares.append((actual, cuenta)); actual, cuenta = v, 1
    pares.append((actual, cuenta))
    return pares

muestra = [r.Country for r in ventas.select("Country").limit(30).collect()]

for titulo, valores in [("Sin ordenar", muestra), ("Ordenados", sorted(muestra))]:
    dicc, indices = diccionario(valores)
    pares = rle(indices)
    bits = max(1, math.ceil(math.log2(len(dicc))))
    print(f"--- {titulo} ---")
    print(f"Diccionario: {dicc}")
    print(f"Índices:     {indices}")
    print(f"RLE:         {pares}  ->  {len(pares)} pares en lugar de {len(indices)} valores")
    print(f"Bit-packing: {bits} bits por índice en lugar de 32\n")

Con los datos ordenados, RLE reduce 30 valores a solo 4 pares. Retén esta idea, porque reaparecerá con el *pushdown*: **el orden en que escribimos los datos es una decisión de diseño, no un detalle.**

En un formato de filas esta codificación es imposible: en cada registro se mezclan cadenas, enteros y fechas, y no hay homogeneidad que aprovechar.

### 3.2. El tamaño importa

Los apuntes citan un análisis sobre AWS Athena, que cobra 5 $ por cada TB escaneado. En él, 1 TB de CSV pasa a ocupar unos 130 GB en Parquet, y cuando una consulta solo necesita una de las columnas, el ahorro en datos escaneados (y, por tanto, en dinero) es todavía mayor.

Vamos a comprobarlo con nuestros datos. Escribimos las ventas en Parquet, con las opciones por defecto, y comparamos los tamaños:

In [ ]:
RUTA_PARQUET = f"{BASE}/ventas_parquet"
ventas.write.mode("overwrite").parquet(RUTA_PARQUET)

t_csv, t_parquet = tamano_bytes(RUTA_CSV), tamano_bytes(RUTA_PARQUET)
print(f"CSV:     {mib(t_csv):7.2f} MiB")
print(f"Parquet: {mib(t_parquet):7.2f} MiB  ({t_csv / t_parquet:.1f} veces más pequeño)")

Ahora supongamos que el analista solo quiere `SELECT SUM(Revenue)`. ¿Cuántos bytes tiene que leer cada formato?

- En **CSV**, aunque solo queramos una columna, hay que recorrer el fichero entero para localizar los separadores.
- En **Parquet**, el pie del fichero indica dónde empieza y cuánto ocupa cada columna. Lo consultamos con **PyArrow**, que puede leer los metadatos de los ficheros que ha escrito Spark:

In [ ]:
import pyarrow.parquet as pq
from collections import defaultdict

def bytes_por_columna(ruta):
    """Suma, por columna, el tamaño comprimido de sus column chunks en todos los ficheros."""
    tamanos = defaultdict(int)
    for fichero in ficheros_datos(ruta, ".parquet"):
        md_ = pq.ParquetFile(fichero).metadata
        for rg in range(md_.num_row_groups):
            for c in range(md_.num_columns):
                col = md_.row_group(rg).column(c)
                tamanos[col.path_in_schema] += col.total_compressed_size
    return dict(tamanos)

columnas = bytes_por_columna(RUTA_PARQUET)
for nombre, b in columnas.items():
    print(f"{nombre:<10} {mib(b):6.2f} MiB")

print(f"\nPara SUM(Revenue): CSV lee {mib(t_csv):.2f} MiB; Parquet lee {mib(columnas['Revenue']):.2f} MiB "
      f"({t_csv / columnas['Revenue']:.0f} veces menos)")

Spark aplica esta optimización automáticamente. En el plan de ejecución, la línea `ReadSchema` indica qué columnas se piden al fichero:

In [ ]:
print("== CSV ==")
plan_scan(ventas_raw.select(F.sum("Revenue")))
print("\n== Parquet ==")
plan_scan(spark.read.parquet(RUTA_PARQUET).select(F.sum("Revenue")))

Las dos lecturas muestran `ReadSchema: struct<Revenue:double>`, pero no significan lo mismo. En CSV, Spark solo **convierte** esa columna, pero **lee del disco todos los bytes**. En Parquet **solo lee del disco** los bytes de esa columna. Las cifras de la celda anterior muestran la diferencia de coste.

## Sección 4. Avro

[Apache Avro](https://avro.apache.org/) es un formato **binario orientado a filas**. Los datos se serializan en un binario compacto y el esquema que los describe se escribe en **JSON**. Esa combinación tiene tres consecuencias:

1. **El esquema viaja con los datos**, en la cabecera del fichero. Quien lee un `.avro` no necesita nada más para interpretarlo.
2. Al estar en JSON, el esquema es **legible y manipulable** con cualquier herramienta.
3. Dos programas pueden leer y escribir el mismo fichero con **versiones distintas del esquema** sin romperse. Esta **evolución de esquema** es lo que distingue a Avro del resto de formatos.

Por eso Avro es la elección natural cuando los datos **cruzan la frontera entre sistemas que evolucionan por separado** (típicamente, un productor y un consumidor de Kafka). Para analizar columnas, el formato adecuado es Parquet.

### 4.1. Estructura del fichero

Un fichero Avro tiene:

- Una **cabecera** con los metadatos: los "números mágicos" `Obj\x01`, el **esquema en JSON** y el **códec** de compresión.
- Una secuencia de **bloques de datos** en binario.
- Un ***sync marker***: 16 bytes aleatorios que aparecen al final de la cabecera y al final de cada bloque.

El *sync marker* explica dos propiedades de la tabla de la Sección 2:

- **Divisible**: un lector puede saltar a cualquier punto del fichero, buscar el siguiente *sync marker* y saber que ahí empieza un bloque completo. Así, varios nodos procesan fragmentos distintos en paralelo.
- **Divisible aunque esté comprimido**: la compresión se aplica bloque a bloque, no al fichero entero.

Los esquemas combinan tipos **primitivos** (`null`, `boolean`, `int`, `long`, `float`, `double`, `bytes`, `string`) y **compuestos** (`record`, `enum`, `array`, `map`, `union`, `fixed`). Este es el esquema de nuestros empleados:

```json
{
  "type": "record",
  "namespace": "com.empresa.rrhh",
  "name": "empleado",
  "fields": [
    {"name": "nombre", "type": "string"},
    {"name": "altura", "type": "int"},
    {"name": "edad",   "type": ["null", "int"], "default": null}
  ]
}
```

> **Campos opcionales.** En Avro todos los campos son obligatorios. Para que `edad` pueda faltar declaramos una **unión** con `null` y un valor `default`. Sin esa unión, escribir un empleado sin edad daría error. La pareja unión + `default` es la base de la evolución de esquema.

### 4.2. Avro en Spark

Databricks incluye el conector Avro de Spark, así que basta con `format("avro")`. Las opciones clave son:

| Opción | Al escribir | Al leer |
|---|---|---|
| `avroSchema` | Fuerza el esquema Avro con el que se escribe | Esquema del **lector** (evolución de esquema) |
| `recordName`, `recordNamespace` | Nombre del registro si Spark genera el esquema | — |
| `compression` | Códec: `uncompressed`, `snappy`, `deflate`, `zstandard`... | — |

Guardamos el esquema en un fichero `.avsc`, igual que haríamos en un proyecto real, y escribimos los empleados con él:

In [ ]:
esquema_v1 = {
    "type": "record", "namespace": "com.empresa.rrhh", "name": "empleado",
    "fields": [
        {"name": "nombre", "type": "string"},
        {"name": "altura", "type": "int"},
        {"name": "edad",   "type": ["null", "int"], "default": None},
    ],
}

RUTA_AVSC = f"{BASE}/empleado.avsc"
with open(RUTA_AVSC, "w") as f:
    json.dump(esquema_v1, f, indent=2)

RUTA_EMP_AVRO = f"{BASE}/empleados_avro"
(empleados.coalesce(1)
    .write.mode("overwrite")
    .format("avro")
    .option("avroSchema", open(RUTA_AVSC).read())
    .save(RUTA_EMP_AVRO))

# Leemos SIN indicar esquema: el fichero ya lo lleva dentro
leidos = spark.read.format("avro").load(RUTA_EMP_AVRO)
leidos.printSchema()
display(leidos)

Spark ha recuperado los tipos sin que le pasemos ningún esquema: Avro es **autodescriptivo**. Para comprobar qué hay realmente dentro del fichero, abrimos su cabecera con `fastavro`:

In [ ]:
import fastavro

fichero_avro = ficheros_datos(RUTA_EMP_AVRO, ".avro")[0]

with open(fichero_avro, "rb") as f:
    lector = fastavro.reader(f)
    print("Esquema almacenado en la cabecera:")
    print(json.dumps(lector.writer_schema, indent=2))
    print(f"\nCódec de compresión: {lector.codec}")
    print(f"Registros: {list(lector)}")

El esquema de la cabecera es nuestro `.avsc`. `fastavro` lo muestra con el nombre completo del registro (`com.empresa.rrhh.empleado`) en lugar de separar `namespace` y `name`, pero es el mismo esquema. El códec es `snappy`, el valor por defecto de Spark para Avro.

Ahora miramos directamente los bytes para localizar los números mágicos y el *sync marker*:

In [ ]:
raw = open(fichero_avro, "rb").read()

print(f"Tamaño total:            {len(raw)} bytes")
print(f"Números mágicos:         {raw[:4]}")

sync = raw[-16:]   # el último bloque termina con el sync marker
print(f"Sync marker:             {sync.hex()}")
print(f"Apariciones del marker:  {raw.count(sync)} (cabecera + un cierre por cada bloque)")

¿Qué pasa si **no** le damos a Spark el esquema? Que lo genera a partir del DataFrame. Escribamos los mismos empleados sin la opción `avroSchema` y veamos qué esquema acaba en la cabecera:

In [ ]:
RUTA_EMP_AVRO_AUTO = f"{BASE}/empleados_avro_auto"
empleados.coalesce(1).write.mode("overwrite").format("avro").save(RUTA_EMP_AVRO_AUTO)

with open(ficheros_datos(RUTA_EMP_AVRO_AUTO, ".avro")[0], "rb") as f:
    print(json.dumps(fastavro.reader(f).writer_schema, indent=2))

Spark ha inventado el nombre `topLevelRecord` y, para las columnas que admiten nulos (al menos `edad`), ha creado uniones del tipo `["int", "null"]` **sin valor `default`**. El fichero es válido, pero ese esquema es un mal punto de partida para un contrato entre sistemas. En producción, **declara siempre el esquema Avro**.

### 4.3. Evolución del esquema

Imagina que llevamos meses escribiendo ficheros de empleados con el esquema v1 y ahora necesitamos un campo `departamento`. Tenemos terabytes de ficheros antiguos sin ese campo. ¿Hay que reescribirlos?

Avro dice que **no**. Para ello distingue dos esquemas:

- ***Writer schema***: el esquema con el que se escribió el fichero. Está en su cabecera.
- ***Reader schema***: el esquema que espera el programa que lee, es decir, la versión actual del contrato.

Cuando no coinciden, Avro aplica la **resolución de esquemas** y traduce de uno a otro al vuelo. En Spark, el esquema del lector se indica con la opción `avroSchema` **al leer**:

In [ ]:
def leer_con(esquema_lector):
    """Lee los ficheros escritos con el esquema v1 usando otro esquema de lector."""
    return (spark.read.format("avro")
            .option("avroSchema", json.dumps(esquema_lector))
            .load(RUTA_EMP_AVRO))

def esquema_con(campos):
    return {"type": "record", "namespace": "com.empresa.rrhh", "name": "empleado", "fields": campos}

NOMBRE = {"name": "nombre", "type": "string"}
ALTURA = {"name": "altura", "type": "int"}
EDAD   = {"name": "edad",   "type": ["null", "int"], "default": None}

**Caso 1. Añadir un campo con valor por defecto.** Los ficheros antiguos se leen sin tocarlos y el campo que les falta se rellena con el `default`:

In [ ]:
v2 = esquema_con([NOMBRE, ALTURA, EDAD,
                  {"name": "departamento", "type": "string", "default": "Sin asignar"}])
display(leer_con(v2))

**Caso 2. Añadir un campo sin valor por defecto.** Avro no puede inventarse un valor que no está en el fichero y para el que no hay alternativa, así que **falla**. Este error es la mejor demostración de para qué sirve un `default`:

In [ ]:
v2b = esquema_con([NOMBRE, ALTURA, EDAD,
                   {"name": "departamento", "type": "string"}])
try:
    leer_con(v2b).collect()
except Exception as e:
    mensaje = str(e)
    pista = [l for l in mensaje.splitlines() if "departamento" in l]
    print(type(e).__name__)
    print(pista[0][:300] if pista else mensaje[:600])

**Caso 3. Renombrar un campo.** Queremos que `nombre` pase a llamarse `nombre_completo`. Para que Avro sepa que es el mismo campo usamos `aliases`:

In [ ]:
v3 = esquema_con([{"name": "nombre_completo", "type": "string", "aliases": ["nombre"]},
                  ALTURA, EDAD])
display(leer_con(v3))

**Caso 4. Eliminar un campo.** Ya no nos interesa la edad. El dato sigue en el fichero, pero el lector lo ignora:

In [ ]:
v4 = esquema_con([NOMBRE, ALTURA])
display(leer_con(v4))

| Cambio en el esquema | ¿Se leen los ficheros antiguos? | Condición |
|---|---|---|
| Añadir un campo | Sí | Debe tener `default` |
| Añadir un campo | **No** | Si no tiene `default` |
| Renombrar un campo | Sí | Mediante `aliases` |
| Eliminar un campo | Sí | Siempre |

Ninguno de estos cambios obliga a reescribir un solo byte de los datos almacenados. Ni Parquet ni ORC ofrecen este nivel de control.

> **Compatibilidad hacia atrás y hacia delante.** Que un lector nuevo lea datos antiguos (lo que acabamos de hacer) es compatibilidad **hacia atrás** (*backward*). Que un lector antiguo lea datos nuevos es compatibilidad **hacia delante** (*forward*), y funciona porque el lector ignora los campos que no conoce. Si se cumplen ambas, la compatibilidad es **completa** (*full*). Añadir un campo con `default` tiene compatibilidad completa. De ahí la regla de oro: **todo campo nuevo lleva `default`**.

> **¿Y si el esquema no cabe en cada mensaje?** En un fichero `.avro` el esquema se escribe una vez y se reparte entre miles de registros. En Kafka cada mensaje es independiente, y repetir el esquema en cada uno multiplicaría su tamaño. La solución es un ***Schema Registry***, que guarda los esquemas y les asigna un identificador. Los mensajes viajan con ese identificador en lugar del esquema, así que ya no son autodescriptivos: sin acceso al registro no se pueden leer. En Spark se trabaja con ellos mediante `from_avro`/`to_avro`. Lo veremos en las sesiones de Kafka.

#### Reto 4.3

Comprueba la compatibilidad **hacia delante**:

1. Crea un DataFrame con dos empleados nuevos que incluya la columna `departamento`.
2. Escríbelo en Avro en una ruta nueva usando el esquema `v2` (opción `avroSchema` al escribir).
3. Léelo con el esquema **v1** (`esquema_v1`) como esquema del lector. ¿Qué ocurre con `departamento`?

In [ ]:
# Reto 4.3 · Compatibilidad HACIA DELANTE: un lector antiguo (v1) lee datos nuevos (v2)

# 1. Dos empleados nuevos, ya con la columna `departamento`
esquema_empleados_v2 = T.StructType([
    T.StructField("nombre",       T.StringType(),  nullable=False),
    T.StructField("altura",       T.IntegerType(), nullable=False),
    T.StructField("edad",         T.IntegerType(), nullable=True),
    T.StructField("departamento", T.StringType(),  nullable=False),
])
empleados_nuevos = spark.createDataFrame(
    [("Laura", 170, 31, "Ventas"), ("Marcos", 185, None, "Tecnología")],
    schema=esquema_empleados_v2,
)

# 2. Lo escribimos en Avro en una ruta nueva con el esquema v2 (el definido en la celda del Caso 1)
RUTA_EMP_AVRO_V2 = f"{BASE}/empleados_avro_v2"
(empleados_nuevos.coalesce(1)
    .write.mode("overwrite")
    .format("avro")
    .option("avroSchema", json.dumps(v2))
    .save(RUTA_EMP_AVRO_V2))

# 3. Lo leemos con el esquema v1 como esquema del lector
leidos_v1 = (spark.read.format("avro")
             .option("avroSchema", json.dumps(esquema_v1))
             .load(RUTA_EMP_AVRO_V2))
leidos_v1.printSchema()
display(leidos_v1)

# Conclusión: el lector antiguo NO conoce `departamento`, así que simplemente lo IGNORA y lee el resto sin error.
# Eso es compatibilidad hacia delante. Como añadir un campo con `default` también permite que un lector nuevo lea
# datos antiguos (hacia atrás), la compatibilidad es completa (full).

### 4.4. Avro con un caso real: ventas de Alemania

Repetimos el ejemplo de los apuntes con PySpark: partimos del CSV de ventas, limpiamos los códigos postales, nos quedamos con Alemania y guardamos el resultado en Avro. Esta vez dejamos que Spark genere el esquema, pero le damos un nombre y un *namespace* significativos. Guardamos dos versiones, con *Snappy* y con *deflate*:

In [ ]:
ventas_alemania = (ventas_raw
                   .withColumn("Zip", F.trim("Zip"))
                   .filter(F.col("Country") == "Germany"))

RUTA_DE_SNAPPY  = f"{BASE}/ventas_de_avro_snappy"
RUTA_DE_DEFLATE = f"{BASE}/ventas_de_avro_deflate"

for ruta, codec in [(RUTA_DE_SNAPPY, "snappy"), (RUTA_DE_DEFLATE, "deflate")]:
    (ventas_alemania.coalesce(1).write.mode("overwrite")
        .format("avro")
        .option("recordName", "Sales")
        .option("recordNamespace", "com.empresa.ventas")
        .option("compression", codec)
        .save(ruta))
    print(f"{codec:<8} {mib(tamano_bytes(ruta)):6.2f} MiB")

with open(ficheros_datos(RUTA_DE_SNAPPY, ".avro")[0], "rb") as f:
    print("\nEsquema generado:")
    print(json.dumps(fastavro.reader(f).writer_schema, indent=2))

Hablaremos de los algoritmos de compresión y de cuándo usar cada uno en la Sección 7, cuando hayamos visto todos los formatos.

## Sección 5. Apache Arrow

[Apache Arrow](https://arrow.apache.org/) es una especificación de **formato columnar en memoria**. Avro, Parquet y ORC definen cómo guardar los datos **en disco**. Arrow define cómo representarlos **en RAM** para que distintas herramientas y lenguajes los compartan sin copiarlos ni serializarlos.

Hoy es la capa de interoperabilidad estándar del ecosistema de datos en Python. La usan Pandas 2.x, Polars, DuckDB y **Spark**, que recurre a Arrow para mover datos entre la JVM y Python en `toPandas()`, `createDataFrame(pandas_df)` y las *Pandas UDFs*.

Sus ventajas principales son:

- ***Zero-copy***: dos procesos pueden leer los mismos datos en memoria sin duplicarlos.
- **Vectorización**: el formato columnar facilita las instrucciones SIMD (*Single Instruction, Multiple Data*), que procesan varios valores en una sola instrucción.
- **Interoperabilidad**: la misma representación sirve en Python, R, Java, C++, Rust...
- **Ecosistema**: es la base de las librerías de lectura y escritura de Parquet, ORC y del formato Feather.

### 5.1. PyArrow

La estructura central de PyArrow es la **tabla** (`pa.Table`): parecida a un DataFrame, pero con tipado estricto y representación columnar.

In [ ]:
import pyarrow as pa

esquema_pa = pa.schema([
    ("nombre", pa.string()),
    ("altura", pa.int32()),
    ("edad",   pa.int32()),
])

tabla_empleados = pa.Table.from_pydict(
    {"nombre": ["Aitor", "Pedro"], "altura": [182, 178], "edad": [48, None]},
    schema=esquema_pa,
)
print(tabla_empleados)

Desde Spark 4.0, un DataFrame de Spark se puede convertir directamente en una tabla Arrow con `toArrow()`. Es la forma más eficiente de llevar un resultado de Spark a la memoria del *driver* para seguir trabajando en Python. En versiones anteriores, se pasa por Pandas:

In [ ]:
inicio = time.perf_counter()
try:
    tabla_ventas = ventas.toArrow()
except Exception:               # Spark < 4.0 o cliente sin toArrow()
    tabla_ventas = pa.Table.from_pandas(ventas.toPandas(), preserve_index=False)
duracion = time.perf_counter() - inicio

print(tabla_ventas.schema)
print(f"\nFilas: {tabla_ventas.num_rows:,} | Memoria: {mib(tabla_ventas.nbytes):.1f} MiB | {duracion:.1f} s")

Observa que `Date` ha llegado como `date32` y `Revenue` como `double`. Los tipos se conservan en el paso de Spark a Arrow, cosa que no ocurría con los antiguos tipos de NumPy.

### 5.2. PyArrow y Pandas

Desde Pandas 2.0 se puede usar Arrow como almacenamiento interno de un DataFrame. Mejora el rendimiento y reduce la memoria, sobre todo con **cadenas de texto y valores nulos**. Leemos el mismo CSV desde el volumen con los dos *backends* y comparamos:

In [ ]:
import pandas as pd

pdf_numpy = pd.read_csv(RUTA_CSV, sep=";")
pdf_arrow = pd.read_csv(RUTA_CSV, sep=";", dtype_backend="pyarrow")

comparativa = pd.DataFrame({
    "tipo (NumPy)": pdf_numpy.dtypes.astype(str),
    "tipo (Arrow)": pdf_arrow.dtypes.astype(str),
})
print(comparativa, "\n")
print(f"Memoria con NumPy: {mib(pdf_numpy.memory_usage(deep=True).sum()):6.1f} MiB")
print(f"Memoria con Arrow: {mib(pdf_arrow.memory_usage(deep=True).sum()):6.1f} MiB")

Las columnas de texto (`Date`, `Zip`, `Country`) son las que más memoria ahorran con Arrow. En las columnas numéricas puras la diferencia es pequeña.

### 5.3. El formato Feather

Además del formato en memoria, Arrow define un formato de fichero, **Feather** (o *Arrow IPC*), cuya estructura en disco es **la misma que en RAM**. Leerlo es prácticamente copiar bytes, así que es ideal para **ficheros intermedios** que se escriben y se leen continuamente dentro de un mismo *pipeline*.

Spark no lee Feather de forma nativa. En Databricks lo usaremos en los pasos que se ejecutan en Python sobre el *driver*:

In [ ]:
import pyarrow.feather as feather

RUTA_FEATHER = f"{BASE}/ventas.feather"

inicio = time.perf_counter()
feather.write_feather(tabla_ventas, RUTA_FEATHER, compression="uncompressed")
t_escritura = time.perf_counter() - inicio

inicio = time.perf_counter()
tabla_leida = feather.read_table(RUTA_FEATHER)
t_lectura = time.perf_counter() - inicio

print(f"Feather: {mib(tamano_bytes(RUTA_FEATHER)):.2f} MiB | "
      f"escritura {t_escritura:.2f} s | lectura {t_lectura:.2f} s")
print(f"¿Son idénticas las tablas? {tabla_leida.equals(tabla_ventas)}")

> **¿Feather o Parquet?** Usa **Feather** para ficheros intermedios que se leen y escriben con frecuencia dentro de un mismo *pipeline*. Usa **Parquet** para el almacenamiento definitivo, sobre todo si lo van a leer Spark, Athena u otras herramientas analíticas. Fíjate en el tamaño: Feather sin comprimir ocupa casi tanto como el CSV, o más. Está pensado para leer rápido, no para ahorrar espacio.

## Sección 6. Apache Parquet

[Apache Parquet](https://parquet.apache.org/) es un formato **orientado a columnas** que admiten prácticamente todos los *frameworks* y lenguajes. Como Avro, es **autodescriptivo**: el esquema va dentro del fichero. Es el formato por defecto de Spark y la base de Delta Lake.

Es idóneo para *datasets* con muchas columnas de las que solo se consultan unas pocas: comprime muy bien y en cada lectura solo recorre las columnas necesarias. Si Avro responde a "*los datos cruzan sistemas que evolucionan por separado*", Parquet responde a "*quiero escanear miles de millones de filas por dos columnas y que me cueste poco*".

### 6.1. Estructura del fichero

Un fichero Parquet se organiza así:

```text
Fichero
├── Row group 0                ← bloque horizontal de filas (unidad de paralelismo)
│   ├── Column chunk: ProductID   ← los valores de UNA columna, contiguos
│   │   ├── Dictionary page       ← el diccionario, si se usa
│   │   ├── Data page             ← unidad de codificación, compresión y lectura
│   │   └── Data page
│   ├── Column chunk: Date
│   └── ...
├── Row group 1
│   └── ...
└── Footer (pie)               ← esquema + posición, codificación, compresión
                                 y estadísticas (min, max, nulos) de cada column chunk
```

- **Row group**: un bloque horizontal de filas. Cada uno lo puede procesar un nodo distinto.
- **Column chunk**: dentro de un *row group*, los valores de una columna almacenados de forma contigua. Permite leer una columna sin tocar las demás.
- **Página**: la subdivisión final. Es la unidad de codificación y de compresión, y la unidad mínima de lectura: aunque solo necesitemos un valor, se lee la página entera.
- **Pie (*footer*)**: los metadatos. Se escribe al final porque las estadísticas solo se conocen cuando se han visto todos los datos, y así el fichero se escribe en una única pasada.

> **La página es la pieza clave.** Como cada página indica su codificación y su compresión, **cada columna puede elegir la que mejor le va**. Cuando se usa diccionario, este se guarda en una página especial al principio del *column chunk* y el resto de páginas solo contienen índices. La compresión se aplica después de codificar, página a página, por lo que Parquet sigue siendo divisible aunque esté comprimido.

### 6.2. Parquet con PySpark

Repetimos el ejemplo de los empleados en Parquet e inspeccionamos el fichero con PyArrow:

In [ ]:
RUTA_EMP_PARQUET = f"{BASE}/empleados_parquet"
empleados.coalesce(1).write.mode("overwrite").parquet(RUTA_EMP_PARQUET)

display(spark.read.parquet(RUTA_EMP_PARQUET))

fichero = ficheros_datos(RUTA_EMP_PARQUET, ".parquet")[0]
print("Esquema guardado en el pie del fichero:")
print(pq.read_schema(fichero))

`altura` y `edad` se han guardado como `int32`, porque así lo declaramos en `esquema_empleados`.

> **Si no indicamos el esquema, Spark se lo inventa.** Al inferir un JSON, Spark asigna `bigint` (64 bits) a cualquier entero, y nadie te avisa de que esas columnas ocupan el doble (es lo que viste en el Reto 2.1). Al inferir un CSV, elige `int` cuando los valores caben, pero deja `Date` como texto, porque no reconoce el formato `M/d/yyyy`, y además tiene que leer el fichero una vez más solo para adivinar los tipos. **Inferir el esquema es cómodo para explorar, pero en un *pipeline* de producción hay que declararlo siempre.**

### 6.3. Controlando los *row groups*

Para que las optimizaciones de las próximas secciones se vean con claridad, escribimos las ventas en **un único fichero, ordenado por país y fecha**, con *row groups* más pequeños de lo habitual. En Spark, el tamaño del *row group* se controla en bytes con la opción `parquet.block.size` (por defecto, 128 MiB):

In [ ]:
RUTA_PARQUET_ORDENADO = f"{BASE}/ventas_parquet_ordenado"

(ventas
    .repartition(1)
    .sortWithinPartitions("Country", "Date")
    .write.mode("overwrite")
    .option("parquet.block.size", 2 * 1024 * 1024)   # row groups de ~2 MiB
    .parquet(RUTA_PARQUET_ORDENADO))

fichero_ordenado = ficheros_datos(RUTA_PARQUET_ORDENADO, ".parquet")[0]
metadatos = pq.ParquetFile(fichero_ordenado).metadata
print(f"Filas: {metadatos.num_rows:,} | Row groups: {metadatos.num_row_groups}")

Spark decide dónde cortar cada *row group* según el tamaño **en memoria** de los datos, así que el número exacto puede variar. Además, en *Serverless* la escritura puede hacerla el motor Photon, que no siempre respeta `parquet.block.size`. Si obtienes **un solo *row group***, ejecuta la celda siguiente: reescribe el fichero con PyArrow, a partir de los mismos datos ordenados, en exactamente 4 *row groups*. Si ya tienes varios, la celda no hace nada.

In [ ]:
# Alternativa (solo si en la celda anterior obtuviste 1 row group)
if metadatos.num_row_groups < 2:
    tabla_ordenada = tabla_ventas.sort_by([("Country", "ascending"), ("Date", "ascending")])
    dbutils.fs.rm(RUTA_PARQUET_ORDENADO, True)
    os.makedirs(RUTA_PARQUET_ORDENADO, exist_ok=True)
    fichero_ordenado = f"{RUTA_PARQUET_ORDENADO}/part-00000.parquet"
    pq.write_table(tabla_ordenada, fichero_ordenado,
                   row_group_size=math.ceil(tabla_ordenada.num_rows / 4))
    metadatos = pq.ParquetFile(fichero_ordenado).metadata
print(f"Row groups: {metadatos.num_row_groups}")

### 6.4. Datos anidados

¿Cómo guarda un formato columnar un pedido que contiene un cliente y una lista de líneas? La solución de Parquet es **aplanar el esquema hasta sus hojas**: cada campo simple del árbol se convierte en una columna física independiente, cuyo nombre es su ruta completa.

In [ ]:
esquema_pedidos = T.StructType([
    T.StructField("order_id", T.LongType()),
    T.StructField("cliente", T.StructType([
        T.StructField("nombre", T.StringType()),
        T.StructField("pais",   T.StringType()),
    ])),
    T.StructField("lineas", T.ArrayType(T.StructType([
        T.StructField("sku", T.StringType()),
        T.StructField("uds", T.IntegerType()),
    ]))),
])

pedidos = spark.createDataFrame([
    (1, ("Acme Corp", "ES"), [("A1", 2), ("B7", 1)]),
    (2, ("Globex",    "FR"), [("C3", 5)]),
], schema=esquema_pedidos)

RUTA_PEDIDOS = f"{BASE}/pedidos_parquet"
pedidos.coalesce(1).write.mode("overwrite").parquet(RUTA_PEDIDOS)

pf = pq.ParquetFile(ficheros_datos(RUTA_PEDIDOS, ".parquet")[0])
print(f"Columnas lógicas: {len(pedidos.columns)} -> Columnas físicas: {pf.metadata.num_columns}")
for i in range(pf.metadata.num_columns):
    print("   ", pf.metadata.row_group(0).column(i).path_in_schema)

Las 3 columnas lógicas se han convertido en 5 columnas físicas. Como cada hoja es una columna de pleno derecho, Spark puede leer **una subcolumna anidada sin leer el resto del pedido** (*nested schema pruning*). Fíjate en `ReadSchema`:

In [ ]:
consulta = spark.read.parquet(RUTA_PEDIDOS).select("order_id", "cliente.nombre")
display(consulta)
plan_scan(consulta)

Spark solo ha pedido `order_id` y `cliente.nombre`. Ni el país del cliente, ni los SKU, ni las unidades se han leído del disco. Por eso Parquet es la mejor opción para documentos anidados de los que solo se consultan algunas subcolumnas.

> **¿Cómo reconstruye Parquet el pedido?** Si `lineas.list.element.sku` contiene `["A1", "B7", "C3"]`, ¿cómo sabe que los dos primeros son del pedido 1 y el tercero del pedido 2? Junto a cada valor guarda dos enteros muy pequeños. El **nivel de repetición** (*repetition level*) indica en qué punto del árbol empieza un elemento nuevo. El **nivel de definición** (*definition level*) indica hasta qué profundidad está definido el valor, y permite distinguir una lista vacía de una lista con un nulo.

### 6.5. Leyendo menos datos

¿Cómo se escanean mil millones de filas en dos segundos? **No escaneándolas.** El mismo truco se aplica a tres niveles, y cada uno amplía el anterior:

1. **Codificación**: que cada valor ocupe menos bytes.
2. ***Pushdown***: no leer las columnas ni los *row groups* que no hacen falta.
3. **Particionado**: no llegar siquiera a abrir el fichero.

Los tres son gratis para quien consulta y transparentes para el desarrollador, **siempre que los datos se hayan escrito con sentido**. Ese "siempre que" es el trabajo del ingeniero de datos.

#### 6.5.1. Codificación

Toda la información de codificación está en el pie del fichero, y se puede leer sin tocar los datos. Inspeccionamos el fichero ordenado: para cada *row group* y columna vemos la compresión, las codificaciones, el tamaño antes y después de comprimir, y las estadísticas.

In [ ]:
def inspeccionar_parquet(fichero, max_row_groups=4):
    md_ = pq.ParquetFile(fichero).metadata
    print(f"Filas: {md_.num_rows:,} | Row groups: {md_.num_row_groups}")
    cabecera = (f"  {'columna':<10} {'compr.':<8} {'codificación':<34} "
                f"{'codific.':>9} {'comprim.':>9} {'ratio':>6}  {'min':<12} {'max':<12} {'nulos':>5}")
    for r in range(min(md_.num_row_groups, max_row_groups)):
        rg = md_.row_group(r)
        print(f"\n--- Row group {r}: {rg.num_rows:,} filas ---")
        print(cabecera)
        for i in range(rg.num_columns):
            col = rg.column(i)
            st = col.statistics
            sin, con = col.total_uncompressed_size, col.total_compressed_size
            linea = (f"  {col.path_in_schema:<10} {str(col.compression):<8} "
                     f"{', '.join(col.encodings):<34} {sin/1024:>6.0f} KiB {con/1024:>6.0f} KiB "
                     f"{sin/con:>5.1f}x  ")
            if st is None or not st.has_min_max:
                print(linea + "(sin estadísticas)")
            else:
                print(linea + f"{str(st.min)[:12]:<12} {str(st.max)[:12]:<12} {st.null_count:>5}")
    if md_.num_row_groups > max_row_groups:
        print(f"\n... y {md_.num_row_groups - max_row_groups} row groups más")

inspeccionar_parquet(fichero_ordenado)

Fíjate en tres cosas:

- La codificación por diccionario (`PLAIN_DICTIONARY` o `RLE_DICTIONARY`, según la versión del escritor) se aplica **automáticamente**, sin pedirla.
- La compresión es la que Spark usa por defecto (`spark.sql.parquet.compression.codec`). Comprueba cuál aparece en tu entorno.
- `Country` apenas ocupa unos pocos KiB por *row group*: con los datos ordenados, diccionario + RLE la reducen a unos pocos pares (valor, repeticiones). Es el efecto que simulamos en la Sección 3.1. Por eso Parquet es especialmente eficiente con columnas de **baja cardinalidad**.

#### 6.5.2. *Pushdown*

Las estadísticas del pie permiten dos optimizaciones que Spark, Athena, DuckDB o PyArrow aplican automáticamente:

- ***Projection pushdown*** (poda de columnas): solo se leen del disco las columnas de la consulta. Es lo que medimos en la Sección 3.2.
- ***Predicate pushdown*** (*data skipping*): antes de leer, el motor compara el filtro con el **mínimo y el máximo** de cada *row group* y descarta los que no pueden contener resultados.

Veamos las estadísticas de `Country` y `Date` en cada *row group*, y razonemos como lo haría el motor ante dos filtros:

In [ ]:
import datetime

def estadisticas(fichero, columna):
    md_ = pq.ParquetFile(fichero).metadata
    idx = md_.schema.names.index(columna)
    return [(md_.row_group(r).num_rows,
             md_.row_group(r).column(idx).statistics.min,
             md_.row_group(r).column(idx).statistics.max)
            for r in range(md_.num_row_groups)]

def simular_skipping(fichero, columna, valor):
    """Decide, solo con min/max del pie, qué row groups habría que leer para columna == valor."""
    total = leidas = 0
    print(f"Filtro: {columna} = {valor}")
    for r, (filas, vmin, vmax) in enumerate(estadisticas(fichero, columna)):
        se_lee = vmin <= valor <= vmax
        total += filas
        leidas += filas if se_lee else 0
        print(f"  RG{r}: min={vmin}  max={vmax}  ->  {'SE LEE' if se_lee else 'descartado'}")
    print(f"  Filas leídas: {leidas:,} de {total:,} ({leidas / total:.0%})\n")

simular_skipping(fichero_ordenado, "Country", "Germany")
simular_skipping(fichero_ordenado, "Date", datetime.date(2010, 3, 15))

En el **mismo fichero**, con el **mismo formato** y la **misma codificación**, el filtrado anticipado funciona muy bien con `Country` y no sirve para nada con `Date`. La diferencia no está en Parquet, sino en cómo se escribió el fichero: **ordenado por `Country`**, así que cada *row group* contiene pocos países, pero las fechas solo están ordenadas dentro de cada país y casi todos los *row groups* abarcan un rango de fechas muy amplio.

Spark aplica este razonamiento él solo. En el plan, `PushedFilters` muestra los filtros que se entregan al lector de Parquet para que descarte *row groups*:

In [ ]:
consulta_alemania = (spark.read.parquet(RUTA_PARQUET_ORDENADO)
                     .filter(F.col("Country") == "Germany")
                     .select("Country", "Revenue"))
plan_scan(consulta_alemania)
display(consulta_alemania.agg(F.round(F.sum("Revenue"), 2).alias("ventas_alemania")))

> **El *pushdown* no es magia.** Para que el filtrado anticipado descarte algo se necesitan dos condiciones:
> 1. Que el fichero tenga **varios *row groups***. Un fichero con un único *row group* nunca se salta nada.
> 2. Que los datos estén **ordenados, o al menos agrupados, por la columna del filtro**. Si los países están mezclados, todos los *row groups* tendrán `min=Canada, max=Mexico` y no se podrá descartar ninguno.

#### Reto 6.5

Demuestra la segunda condición con datos reales:

1. Recorre los ficheros de `RUTA_PARQUET` (escrito en la Sección 3.2 **sin ordenar**) con `ficheros_datos()` e inspecciona la columna `Country` de cada uno con `estadisticas()`. ¿Cuántos *row groups* podría descartar el filtro `Country = 'Germany'`?
2. Escribe una nueva versión del dataset **ordenada por `Date`** (con `parquet.block.size` pequeño) y usa `simular_skipping()` para comprobar qué filtro funciona ahora y cuál deja de funcionar.

*Pista:* si Spark te genera un único *row group*, usa la misma técnica que la celda alternativa de la Sección 6.3, pero ordenando la tabla Arrow por `Date`.

In [ ]:
# Reto 6.5 · ¿Cuándo funciona el data skipping? Depende del ORDEN con que se escribieron los datos

# 1. Fichero(s) SIN ordenar (Sección 3.2): estadísticas de Country en cada row group
print("=== Parquet SIN ordenar ===")
for f in ficheros_datos(RUTA_PARQUET, ".parquet"):
    print(os.path.basename(f))
    simular_skipping(f, "Country", "Germany")
# Cada row group contiene los 4 países mezclados: min=Canada y max=Mexico en todos,
# así que el filtro Country = 'Germany' no puede descartar NINGÚN row group.

# 2. Nueva versión ordenada por Date
RUTA_PARQUET_FECHA = f"{BASE}/ventas_parquet_por_fecha"
(ventas
    .repartition(1)
    .sortWithinPartitions("Date")
    .write.mode("overwrite")
    .option("parquet.block.size", 2 * 1024 * 1024)
    .parquet(RUTA_PARQUET_FECHA))

fichero_fecha = ficheros_datos(RUTA_PARQUET_FECHA, ".parquet")[0]
if pq.ParquetFile(fichero_fecha).metadata.num_row_groups < 2:
    # Misma técnica que en la Sección 6.3: reescribimos con PyArrow en 4 row groups
    tabla_fecha = tabla_ventas.sort_by([("Date", "ascending")])
    dbutils.fs.rm(RUTA_PARQUET_FECHA, True)
    os.makedirs(RUTA_PARQUET_FECHA, exist_ok=True)
    fichero_fecha = f"{RUTA_PARQUET_FECHA}/part-00000.parquet"
    pq.write_table(tabla_fecha, fichero_fecha, row_group_size=math.ceil(tabla_fecha.num_rows / 4))
print(f"\nRow groups del fichero ordenado por Date: {pq.ParquetFile(fichero_fecha).metadata.num_row_groups}\n")

print("=== Parquet ordenado por Date ===")
simular_skipping(fichero_fecha, "Date", datetime.date(2010, 3, 15))   # ahora SÍ descarta row groups
simular_skipping(fichero_fecha, "Country", "Germany")                  # ahora NO descarta ninguno

# Conclusión: el skipping solo funciona sobre la columna por la que están ordenados (o agrupados) los datos.
# Ordenado por Country, el filtro por país funcionaba y el de fecha no; ordenado por Date, ocurre justo lo contrario.
# El orden de escritura es una decisión de diseño: hay que ordenar por la columna por la que más se filtra.

#### 6.5.3. Particionado

En la práctica, una tabla Parquet casi nunca es un fichero, sino **un directorio**. El descarte por estadísticas tiene un límite: el motor tiene que abrir cada fichero y leer su pie. Si repartimos los datos en carpetas según el valor de una columna, el motor puede **descartar directorios enteros sin abrir ningún fichero**. Es la misma idea, un nivel más arriba.

En Spark se hace con `partitionBy`:

In [ ]:
RUTA_PARTICIONADO = f"{BASE}/ventas_particionado"
ventas.write.mode("overwrite").partitionBy("Country").parquet(RUTA_PARTICIONADO)

arbol(RUTA_PARTICIONADO)

Los directorios siguen la convención `columna=valor`, conocida como ***Hive-style partitioning***, que entienden prácticamente todos los motores del ecosistema. Para leer, basta con apuntar al directorio raíz. El filtro sobre la columna de partición aparece en el plan como `PartitionFilters`:

In [ ]:
consulta_part = (spark.read.parquet(RUTA_PARTICIONADO)
                 .filter(F.col("Country") == "Germany")
                 .select("Country", "Revenue"))
plan_scan(consulta_part)
display(consulta_part.agg(F.round(F.sum("Revenue"), 2).alias("ventas_alemania")))

Un detalle elegante: **dentro de `Country=Germany/` los ficheros ya no contienen la columna `Country`**. Repetir "Germany" en cada fila no tendría sentido: el valor está en el nombre del directorio y el motor lo reconstruye al leer. Compruébalo:

In [ ]:
fichero_alemania = ficheros_datos(f"{RUTA_PARTICIONADO}/Country=Germany", ".parquet")[0]
print(pq.read_schema(fichero_alemania))

Es la codificación por diccionario llevada al sistema de ficheros: la partición cuesta cero bytes de datos.

Elegir la columna de particionado es una de las decisiones de diseño más importantes de un *data lake*. Hay que particionar por la columna por la que se filtra (típicamente, la fecha) **sin pasarse**: miles de directorios con un fichero diminuto cada uno empeoran el rendimiento en lugar de mejorarlo. Lo pondremos en práctica en la sesión sobre almacenamiento distribuido.

> **¿Y Apache ORC?** [ORC](https://orc.apache.org/) (*Optimized Row Columnar*) nació en 2013, el mismo año que Parquet, con el mismo objetivo y un diseño casi idéntico. Los bloques de filas se llaman *stripes* en lugar de *row groups*, y también tiene codificación por columna, compresión por bloque y estadísticas para el *pushdown*. Fue el formato nativo de Hive, pero hoy Parquet se ha impuesto fuera de ese ecosistema. ORC sigue siendo relevante en tablas Hive transaccionales y en migraciones desde ellas. Todo lo que has aprendido de Parquet se aplica a ORC, y en Spark se usa igual:

In [ ]:
RUTA_ORC = f"{BASE}/ventas_orc"
ventas.write.mode("overwrite").orc(RUTA_ORC)

print(f"ORC: {mib(tamano_bytes(RUTA_ORC)):.2f} MiB")
plan_scan(spark.read.orc(RUTA_ORC).filter(F.col("Country") == "Germany").select("Revenue"))

## Sección 7. Compresión

Comprimir hace que los datos ocupen menos en el almacenamiento (y cuesten menos), que se lean más rápido del disco y que viajen menos bytes por la red. A cambio, comprimir y descomprimir consume CPU.

> **Compresión ≠ codificación.** Recuerda la Sección 3.1: Parquet y ORC comprimen mejor que un CSV, no porque usen mejores algoritmos, sino porque **llegan al compresor con los datos ya recodificados**.

### 7.1. Eligiendo el algoritmo

Durante una década, elegir códec fue un dilema: rápido (***Snappy***) o pequeño (***gzip***), pero no ambas cosas. Por eso *Snappy* acabó siendo el códec por defecto de Parquet: en un clúster, cada ciclo de CPU dedicado a descomprimir se le quita a la consulta. ***Zstandard*** (*zstd*) rompió ese equilibrio: ofrece una velocidad comparable a *Snappy* con ratios de compresión parecidos a *gzip*.

Cuidado con una trampa práctica: **el mismo algoritmo cambia de nombre según el formato**. En Spark, todos se indican con la opción `compression` del escritor:

| Algoritmo | CSV / JSON | Avro | Parquet | ORC | Velocidad | Compresión | ¿Cuándo? |
|---|---|---|---|---|---|---|---|
| *Snappy* | `snappy` | `snappy` | `snappy` | `snappy` | Alta | Baja | Valor por defecto histórico de Parquet |
| *LZ4* | `lz4` | — | `lz4` | `lz4` | Muy alta | Baja | Cuando solo importa la latencia de lectura |
| *DEFLATE* | `gzip`, `deflate` | `deflate` | `gzip` | `zlib` | Baja | Media | Interoperabilidad con herramientas antiguas |
| *Zstandard* | `zstd` | `zstandard` | `zstd` | `zstd` | Alta | Alta | **La opción razonable por defecto hoy** |

### 7.2. Comparativa: formato × códec

Escribimos las mismas ventas en cada combinación de formato y códec, medimos el tamaño y el tiempo de escritura, y lo comparamos con el CSV de partida. Usamos `coalesce(1)` para que todas las salidas sean un único fichero comparable. Feather se escribe con PyArrow desde la tabla Arrow de la Sección 5.

La celda tarda unos minutos en ejecutarse. Si algún códec no está disponible en tu entorno, la celda lo indica y continúa con el resto.

In [ ]:
RUTA_COMPRESION = f"{BASE}/compresion"

experimentos = [
    ("CSV",     "none"),   ("CSV",     "gzip"),
    ("JSONL",   "none"),   ("JSONL",   "gzip"),
    ("Avro",    "uncompressed"), ("Avro", "snappy"), ("Avro", "deflate"), ("Avro", "zstandard"),
    ("Parquet", "none"),   ("Parquet", "snappy"), ("Parquet", "gzip"), ("Parquet", "zstd"),
    ("ORC",     "none"),   ("ORC",     "snappy"), ("ORC",     "zlib"), ("ORC",     "zstd"),
]
formato_spark = {"CSV": "csv", "JSONL": "json", "Avro": "avro", "Parquet": "parquet", "ORC": "orc"}

tamano_csv = tamano_bytes(RUTA_CSV)
resultados = []

for formato, codec in experimentos:
    ruta = f"{RUTA_COMPRESION}/{formato.lower()}_{codec}"
    escritor = (ventas.coalesce(1).write.mode("overwrite")
                .format(formato_spark[formato])
                .option("compression", codec))
    if formato == "CSV":
        escritor = escritor.option("header", True).option("sep", ";")
    inicio = time.perf_counter()
    try:
        escritor.save(ruta)
    except Exception as e:                       # un códec no disponible no detiene el experimento
        print(f"{formato:<8} {codec:<13} NO DISPONIBLE: {str(e).splitlines()[0][:120]}")
        continue
    segundos = time.perf_counter() - inicio
    resultados.append((formato, codec, mib(tamano_bytes(ruta)), segundos))
    print(f"{formato:<8} {codec:<13} listo")

for codec in ["uncompressed", "zstd"]:
    ruta = f"{RUTA_COMPRESION}/feather_{codec}.feather"
    inicio = time.perf_counter()
    feather.write_feather(tabla_ventas, ruta, compression=codec)
    resultados.append(("Feather", codec, mib(tamano_bytes(ruta)), time.perf_counter() - inicio))

tabla_resultados = pd.DataFrame(resultados, columns=["formato", "compresion", "tamano_mib", "escritura_s"])
tabla_resultados["veces_menor_que_csv"] = (mib(tamano_csv) / tabla_resultados["tamano_mib"]).round(1)
tabla_resultados = tabla_resultados.round({"tamano_mib": 2, "escritura_s": 1})

print(f"\nFichero de partida: pdi_sales.csv · {mib(tamano_csv):.2f} MiB")
display(tabla_resultados.sort_values("tamano_mib"))

Busca en tu tabla el detalle más revelador: **Parquet sin comprimir ocupa menos que Avro comprimido con *Snappy***. La codificación columnar consigue por sí sola más reducción que la compresión de un formato de filas. La compresión ayuda, pero **la decisión importante es el formato**.

Fíjate también en JSONL: repetir los nombres de las claves en cada registro lo convierte en el formato más voluminoso.

Por último, medimos el coste de **leer** con una consulta analítica típica sobre tres de las salidas. Los tiempos dependen de la carga del cómputo *serverless* y de sus cachés, así que ejecútala dos o tres veces y fíjate en el orden de magnitud, no en las décimas:

In [ ]:
lectores = {
    "CSV (none)":     lambda: spark.read.option("header", True).option("sep", ";")
                                   .schema(ventas.schema).csv(f"{RUTA_COMPRESION}/csv_none"),
    "Avro (snappy)":  lambda: spark.read.format("avro").load(f"{RUTA_COMPRESION}/avro_snappy"),
    "Parquet (zstd)": lambda: spark.read.parquet(f"{RUTA_COMPRESION}/parquet_zstd"),
}

for nombre, leer in lectores.items():
    inicio = time.perf_counter()
    try:
        total = (leer().filter(F.col("Country") == "Germany")
                       .agg(F.sum("Revenue")).collect()[0][0])
    except Exception as e:
        print(f"{nombre:<15} no se pudo leer: {str(e).splitlines()[0][:120]}")
        continue
    print(f"{nombre:<15} {time.perf_counter() - inicio:5.2f} s   SUM(Revenue) = {total:,.1f}")

### 7.3. Entonces, ¿qué formato elijo?

| Situación | Formato |
|---|---|
| Lo va a abrir una persona o una hoja de cálculo | **CSV** |
| Es un fichero intermedio dentro del mismo *pipeline* en Python | **Arrow / Feather** |
| Viaja entre sistemas que evolucionan por separado (típicamente Kafka) | **Avro** |
| Es para analítica | **Parquet con *zstd*** |

#### Reto 7.3

¿Influye el **orden** de los datos en la compresión? Escribe las ventas en Parquet con *zstd* dos veces, con `coalesce(1)` en ambos casos:

1. Sin ordenar.
2. Ordenadas con `sortWithinPartitions("Country", "Date", "ProductID")`.

Compara el tamaño total con `tamano_bytes()` y el tamaño de cada columna con `bytes_por_columna()`. ¿Qué columnas se benefician más del orden y por qué? Relaciónalo con el RLE de la Sección 3.1.

In [ ]:
# Reto 7.3 · ¿Influye el orden de los datos en la compresión?
def escribir_parquet_zstd(df, ruta, orden=None):
    df = df.coalesce(1)
    if orden:
        df = df.sortWithinPartitions(*orden)
    df.write.mode("overwrite").option("compression", "zstd").parquet(ruta)

RUTA_ZSTD_SIN = f"{BASE}/reto73_sin_ordenar"
RUTA_ZSTD_ORD = f"{BASE}/reto73_ordenado"

escribir_parquet_zstd(ventas, RUTA_ZSTD_SIN)
escribir_parquet_zstd(ventas, RUTA_ZSTD_ORD, orden=["Country", "Date", "ProductID"])

t_sin, t_ord = tamano_bytes(RUTA_ZSTD_SIN), tamano_bytes(RUTA_ZSTD_ORD)
print(f"Sin ordenar: {mib(t_sin):6.2f} MiB")
print(f"Ordenado:    {mib(t_ord):6.2f} MiB  ({100 * (1 - t_ord / t_sin):.1f}% menos)\n")

col_sin = bytes_por_columna(RUTA_ZSTD_SIN)
col_ord = bytes_por_columna(RUTA_ZSTD_ORD)
comparativa_orden = pd.DataFrame({
    "sin_ordenar_MiB": {c: mib(b) for c, b in col_sin.items()},
    "ordenado_MiB":    {c: mib(col_ord[c]) for c in col_sin},
}).round(3)
comparativa_orden["ahorro_%"] = (100 * (1 - comparativa_orden["ordenado_MiB"] / comparativa_orden["sin_ordenar_MiB"])).round(1)
display(comparativa_orden)

# Conclusión: el orden no cambia el contenido, pero sí la compresión. Se benefician casi solo las columnas por las
# que se ordena primero: Country y Date (~99% menos), porque sus valores iguales quedan consecutivos y RLE/delta las
# reducen a unos pocos pares. ProductID, Zip, Units y Revenue casi no cambian: dentro de cada país y fecha sus
# valores siguen siendo casi aleatorios. En total el fichero baja un ~19%, solo por escribirlo ordenado.

## Sección 8. Limpieza (opcional)

Si quieres liberar espacio cuando termines, borra todo lo que ha generado la práctica. Descomenta las dos líneas y ejecuta la celda. **Esta acción no se puede deshacer.**

In [ ]:
# dbutils.fs.rm(BASE, True)
# spark.sql(f"DROP VOLUME IF EXISTS {CATALOGO}.{ESQUEMA}.{VOLUMEN}")